## Random Forest

### 1. Import libraries

In [40]:
import glob

import numpy as np
import pandas as pandas

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

#### 2. Data preprocessing

##### 2a. Import CSV files and combine into one dataframe

In [2]:
csv_folder = "../data/loose_csv/*.csv"
file_paths = glob.glob(csv_folder)

print(f"Found {len(file_paths)} files")

dataframe_list = [pandas.read_csv(file) for file in file_paths]
dataframe_combined = pandas.concat(dataframe_list, ignore_index=True)
dataframe_combined = dataframe_combined.reset_index(drop=True)

Found 15 files


##### 2b. Overview of data

In [3]:
dataframe_combined.shape

(3385313, 85)

In [4]:
dataframe_combined.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Attack Name,Label
0,34.173.20.6-192.168.137.234-443-58644-6,34.173.20.6,443,192.168.137.234,58644,6,14/09/2023 09:24:26 AM,14746364,6,0,...,0.0,0.0,0.0,0.0,14681116.0,0.000000,14681116.0,14681116.0,Benign Traffic,0
1,34.173.20.6-192.168.137.234-443-58646-6,34.173.20.6,443,192.168.137.234,58646,6,14/09/2023 09:24:41 AM,60102110,18,0,...,219123.0,0.0,219123.0,219123.0,14957706.0,190543.248665,15061923.0,14672051.0,Benign Traffic,0
2,52.40.210.103-192.168.137.40-18665-60378-6,52.40.210.103,18665,192.168.137.40,60378,6,14/09/2023 09:24:14 AM,90106769,9,0,...,582529.0,0.0,582529.0,582529.0,29841412.0,319090.538141,30049066.0,29473997.0,Benign Traffic,0
3,10.0.0.3-10.0.0.254-44505-1883-6,10.0.0.3,44505,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998980,123,123,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,0
4,10.0.0.7-10.0.0.254-45727-1883-6,10.0.0.7,45727,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998539,123,123,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,0


In [5]:
dataframe_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 3385313 entries, 0 to 3385312
Data columns (total 85 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     str    
 1   Src IP                      str    
 2   Src Port                    int64  
 3   Dst IP                      str    
 4   Dst Port                    int64  
 5   Protocol                    int64  
 6   Timestamp                   str    
 7   Flow Duration               int64  
 8   Total Fwd Packet            int64  
 9   Total Bwd packets           int64  
 10  Total Length of Fwd Packet  float64
 11  Total Length of Bwd Packet  float64
 12  Fwd Packet Length Max       float64
 13  Fwd Packet Length Min       float64
 14  Fwd Packet Length Mean      float64
 15  Fwd Packet Length Std       float64
 16  Bwd Packet Length Max       float64
 17  Bwd Packet Length Min       float64
 18  Bwd Packet Length Mean      float64
 19  Bwd Packet Length Std       floa

In [6]:
dataframe_combined["Attack Name"].value_counts()

Attack Name
DoS TCP Flood               2106916
Recon Port Scan              485522
MQTT DDoS Publish Flood      413913
MQTT DoS Connect Flood       238031
Recon OS Scan                 85317
Benign Traffic                32620
Recon Vulnerability Scan       8321
DoS UDP Flood                  3115
DDoS UDP Flood                 2576
DDoS ICMP Flood                2552
MQTT Malformed                 2246
DoS ICMP Flood                 2107
MITM ARP Spoofing              1053
MQTT DoS Publish Flood          953
Recon Ping Sweep                 71
Name: count, dtype: int64

##### 2c. Replace spaces in columns with "_"

In [7]:
dataframe_combined.columns = dataframe_combined.columns.str.replace(" ", "_")
dataframe_combined.columns.to_list()

['Flow_ID',
 'Src_IP',
 'Src_Port',
 'Dst_IP',
 'Dst_Port',
 'Protocol',
 'Timestamp',
 'Flow_Duration',
 'Total_Fwd_Packet',
 'Total_Bwd_packets',
 'Total_Length_of_Fwd_Packet',
 'Total_Length_of_Bwd_Packet',
 'Fwd_Packet_Length_Max',
 'Fwd_Packet_Length_Min',
 'Fwd_Packet_Length_Mean',
 'Fwd_Packet_Length_Std',
 'Bwd_Packet_Length_Max',
 'Bwd_Packet_Length_Min',
 'Bwd_Packet_Length_Mean',
 'Bwd_Packet_Length_Std',
 'Flow_Bytes/s',
 'Flow_Packets/s',
 'Flow_IAT_Mean',
 'Flow_IAT_Std',
 'Flow_IAT_Max',
 'Flow_IAT_Min',
 'Fwd_IAT_Total',
 'Fwd_IAT_Mean',
 'Fwd_IAT_Std',
 'Fwd_IAT_Max',
 'Fwd_IAT_Min',
 'Bwd_IAT_Total',
 'Bwd_IAT_Mean',
 'Bwd_IAT_Std',
 'Bwd_IAT_Max',
 'Bwd_IAT_Min',
 'Fwd_PSH_Flags',
 'Bwd_PSH_Flags',
 'Fwd_URG_Flags',
 'Bwd_URG_Flags',
 'Fwd_Header_Length',
 'Bwd_Header_Length',
 'Fwd_Packets/s',
 'Bwd_Packets/s',
 'Packet_Length_Min',
 'Packet_Length_Max',
 'Packet_Length_Mean',
 'Packet_Length_Std',
 'Packet_Length_Variance',
 'FIN_Flag_Count',
 'SYN_Flag_Count',
 'R

##### 2d. Drop unimportant columns

In [8]:
dataframe_combined = dataframe_combined.drop(columns=["Flow_ID", "Src_IP", "Src_Port", "Dst_IP", "Dst_Port", "Protocol", "Timestamp"])
dataframe_combined.columns.to_list()

['Flow_Duration',
 'Total_Fwd_Packet',
 'Total_Bwd_packets',
 'Total_Length_of_Fwd_Packet',
 'Total_Length_of_Bwd_Packet',
 'Fwd_Packet_Length_Max',
 'Fwd_Packet_Length_Min',
 'Fwd_Packet_Length_Mean',
 'Fwd_Packet_Length_Std',
 'Bwd_Packet_Length_Max',
 'Bwd_Packet_Length_Min',
 'Bwd_Packet_Length_Mean',
 'Bwd_Packet_Length_Std',
 'Flow_Bytes/s',
 'Flow_Packets/s',
 'Flow_IAT_Mean',
 'Flow_IAT_Std',
 'Flow_IAT_Max',
 'Flow_IAT_Min',
 'Fwd_IAT_Total',
 'Fwd_IAT_Mean',
 'Fwd_IAT_Std',
 'Fwd_IAT_Max',
 'Fwd_IAT_Min',
 'Bwd_IAT_Total',
 'Bwd_IAT_Mean',
 'Bwd_IAT_Std',
 'Bwd_IAT_Max',
 'Bwd_IAT_Min',
 'Fwd_PSH_Flags',
 'Bwd_PSH_Flags',
 'Fwd_URG_Flags',
 'Bwd_URG_Flags',
 'Fwd_Header_Length',
 'Bwd_Header_Length',
 'Fwd_Packets/s',
 'Bwd_Packets/s',
 'Packet_Length_Min',
 'Packet_Length_Max',
 'Packet_Length_Mean',
 'Packet_Length_Std',
 'Packet_Length_Variance',
 'FIN_Flag_Count',
 'SYN_Flag_Count',
 'RST_Flag_Count',
 'PSH_Flag_Count',
 'ACK_Flag_Count',
 'URG_Flag_Count',
 'CWR_Flag_Cou

##### 2e. Change 'Label' column type to bool

In [9]:
dataframe_combined["Label"] = dataframe_combined["Label"].astype(bool)
dataframe_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 3385313 entries, 0 to 3385312
Data columns (total 78 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow_Duration               int64  
 1   Total_Fwd_Packet            int64  
 2   Total_Bwd_packets           int64  
 3   Total_Length_of_Fwd_Packet  float64
 4   Total_Length_of_Bwd_Packet  float64
 5   Fwd_Packet_Length_Max       float64
 6   Fwd_Packet_Length_Min       float64
 7   Fwd_Packet_Length_Mean      float64
 8   Fwd_Packet_Length_Std       float64
 9   Bwd_Packet_Length_Max       float64
 10  Bwd_Packet_Length_Min       float64
 11  Bwd_Packet_Length_Mean      float64
 12  Bwd_Packet_Length_Std       float64
 13  Flow_Bytes/s                float64
 14  Flow_Packets/s              float64
 15  Flow_IAT_Mean               float64
 16  Flow_IAT_Std                float64
 17  Flow_IAT_Max                float64
 18  Flow_IAT_Min                float64
 19  Fwd_IAT_Total               floa

##### 2f. Drop rows with null values (currently unnecessary as no rows have null values)

In [12]:
dataframe_combined.dropna()

,Flow_Duration,Total_Fwd_Packet,Total_Bwd_packets,Total_Length_of_Fwd_Packet,Total_Length_of_Bwd_Packet,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,Bwd_Packet_Length_Max,...,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Attack_Name,Label
0,14746364,6,0,63.0,0.0,39.0,0.0,10.500000,16.944025,0.0,...,0.0,0.0,0.0,0.0,14681116.0,0.000000,14681116.0,14681116.0,Benign Traffic,False
1,60102110,18,0,4135.0,0.0,1408.0,0.0,229.722222,490.460505,0.0,...,219123.0,0.0,219123.0,219123.0,14957706.0,190543.248665,15061923.0,14672051.0,Benign Traffic,False
2,90106769,9,0,1164.0,0.0,194.0,0.0,129.333333,97.000000,0.0,...,582529.0,0.0,582529.0,582529.0,29841412.0,319090.538141,30049066.0,29473997.0,Benign Traffic,False
3,119998980,123,123,2157.0,4.0,20.0,0.0,17.536585,2.309257,2.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,False
4,119998539,123,123,5411.0,4.0,46.0,0.0,43.991870,5.699433,2.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3385308,0,2,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,True
3385309,1552,2,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,True
3385310,22717,2,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,True
3385311,0,2,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,MQTT DDoS Publish Flood,True


### 3. Prepare data for model training

##### 3a. Sample smaller subset

In [32]:
def sample_or_take_all(group):
    sample_size = min(len(group), 3000)
    return group.sample(n=sample_size, random_state=42)

small_df = dataframe_combined.groupby('Attack_Name').apply(sample_or_take_all)
small_df = small_df.reset_index(level=0)

In [33]:
small_df.info()

<class 'pandas.DataFrame'>
Index: 35558 entries, 6528 to 2731413
Data columns (total 78 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Attack_Name                 35558 non-null  str    
 1   Flow_Duration               35558 non-null  int64  
 2   Total_Fwd_Packet            35558 non-null  int64  
 3   Total_Bwd_packets           35558 non-null  int64  
 4   Total_Length_of_Fwd_Packet  35558 non-null  float64
 5   Total_Length_of_Bwd_Packet  35558 non-null  float64
 6   Fwd_Packet_Length_Max       35558 non-null  float64
 7   Fwd_Packet_Length_Min       35558 non-null  float64
 8   Fwd_Packet_Length_Mean      35558 non-null  float64
 9   Fwd_Packet_Length_Std       35558 non-null  float64
 10  Bwd_Packet_Length_Max       35558 non-null  float64
 11  Bwd_Packet_Length_Min       35558 non-null  float64
 12  Bwd_Packet_Length_Mean      35558 non-null  float64
 13  Bwd_Packet_Length_Std       35558 non-null

##### 3b. Shuffle samples

In [36]:
small_df_rnd = small_df.sample(frac=1, random_state=42)
small_df_rnd = small_df_rnd.reset_index(drop=True)

In [37]:
small_df_rnd.info()

<class 'pandas.DataFrame'>
RangeIndex: 35558 entries, 0 to 35557
Data columns (total 78 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Attack_Name                 35558 non-null  str    
 1   Flow_Duration               35558 non-null  int64  
 2   Total_Fwd_Packet            35558 non-null  int64  
 3   Total_Bwd_packets           35558 non-null  int64  
 4   Total_Length_of_Fwd_Packet  35558 non-null  float64
 5   Total_Length_of_Bwd_Packet  35558 non-null  float64
 6   Fwd_Packet_Length_Max       35558 non-null  float64
 7   Fwd_Packet_Length_Min       35558 non-null  float64
 8   Fwd_Packet_Length_Mean      35558 non-null  float64
 9   Fwd_Packet_Length_Std       35558 non-null  float64
 10  Bwd_Packet_Length_Max       35558 non-null  float64
 11  Bwd_Packet_Length_Min       35558 non-null  float64
 12  Bwd_Packet_Length_Mean      35558 non-null  float64
 13  Bwd_Packet_Length_Std       35558 non-null

##### 3c. Separate targets and features

In [53]:
features = small_df_rnd.drop(columns=['Attack_Name'])
targets = small_df_rnd['Attack_Name']

##### 3d. Create scaler pipeline

In [54]:
# scaler = StandardScaler()
# scaler.fit_transform(features)

scaler_pipeline = Pipeline(steps=["scaler", StandardScaler()])

### 4. Split data into training and test

In [55]:
features_train, features_test, targets_train, targets_test = train_test_split(features, targets, test_size=0.2, random_state=12, shuffle=True)

### 5. Model Training

##### 5a. Create helper evaluate function

In [56]:
def evaluate(model, features_data, targets_data):
    prediction = model.predict(features_data)
    return {
        "MAE": mean_absolute_error(targets_data, prediction),
        "RMSE": np.sqrt(mean_squared_error(targets_data, prediction)),
        "R2": r2_score(targets_data, prediction)
    }

##### 5b. Split training and validation data

In [57]:
X_train, X_val, Y_train, Y_val = train_test_split(features_train, targets_train, test_size=0.2, random_state=7, shuffle=True)

##### 5c. Create a raw model

In [58]:
model_holdout = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier())
])

##### 5d. Train and evaluate model using fit() and predict()

In [59]:
model_holdout.fit(X_train, Y_train)

holdout_scores = pandas.DataFrame({
    "Train": evaluate(model_holdout, X_train, Y_train),
    "Validation": evaluate(model_holdout, X_val, Y_val),
}).T
holdout_scores

ValueError: could not convert string to float: 'Recon OS Scan'